> I have a main global logger variable for my main python script A. But there is an independent logger for another script B. Now I will use script B in script A and the script B's logger variable is shared from script A. How can I do it? Both script initialization is: logger = setup_logger(LOG_DIR) in global space.

To share the logger from Script A to Script B, you need to handle a common Python behavior: when Script A imports Script B, any global code in Script B (like `logger = setup_logger(LOG_DIR)`) runs immediately.

Here are the two best ways to handle this. **Method 1** is the cleanest, standard Python way. **Method 2** is the literal way to share the variable if you want to keep your current setup.

---

## Method 1: The Standard Python Way (Recommended)

Python’s built-in `logging` module is designed as a singleton registry. Instead of passing a logger variable around, you should configure the logger *once* in Script A, and have Script B simply grab that same logger by name.

### Script B (`script_b.py`)

Modify Script B so it only initializes its own logger if it's run by itself. If it's imported, it fetches the existing logger.

```python
import logging

# Fallback: If imported, it uses Python's logging hierarchy.
# If run directly as main, it sets up its own independent logger.
if __name__ == "__main__":
    from your_config_module import setup_logger
    logger = setup_logger("LOG_DIR_B")
else:
    # This grabs the logger configured by Script A
    logger = logging.getLogger("main_logger") 

def do_something():
    logger.info("Hello from Script B!")

```

### Script A (`script_a.py`)

In your main script, name your logger when you set it up so Script B can find it.

```python
import logging
from your_config_module import setup_logger
import script_b

# 1. Initialize and name the logger in global space
# (Modify your setup_logger to accept/set a name, e.g., logging.getLogger("main_logger"))
logger = setup_logger("LOG_DIR_A") 

def main():
    logger.info("Starting Script A...")
    script_b.do_something()

if __name__ == "__main__":
    main()

```

---

## Method 2: Direct Variable Sharing (The Quick Fix)

If you cannot change how `setup_logger` works and absolutely must pass the exact `logger` variable from Script A into Script B, you need to prevent Script B from running `setup_logger` on import, and then explicitly assign Script A's logger to it.

### Script B (`script_b.py`)

Use the `if __name__ == "__main__":` guard to stop the logger from initializing automatically when Script A imports it.

```python
# Initialize as None globally so it exists as a variable
logger = None 

def do_something():
    # Ensure logger has been assigned before using it
    if logger:
        logger.info("Hello from Script B using Script A's logger!")
    else:
        print("Logger not initialized!")

if __name__ == "__main__":
    # This ONLY runs if you execute script_b.py directly
    from your_config_module import setup_logger
    logger = setup_logger("LOG_DIR_B")
    do_something()

```

### Script A (`script_a.py`)

Import Script B, initialize your logger, and then inject Script A's logger directly into Script B's namespace.

```python
from your_config_module import setup_logger
import script_b

# 1. Initialize Script A's logger
logger = setup_logger("LOG_DIR_A")

# 2. Share the logger variable with Script B
script_b.logger = logger 

def main():
    logger.info("Starting Script A...")
    script_b.do_something() # Script B will now use Script A's logger

if __name__ == "__main__":
    main()

```

### Why Method 1 is preferred:

Method 2 relies on "monkey patching" (overwriting a module's variable from the outside). While it works perfectly for simple scripts, Method 1 scales much better if you decide to add a Script C, D, or E down the line, as they can all look up the same logger natively via `logging.getLogger("main_logger")`.

> My setup_logger is:
>
> ```python
> from utils.logger import setup_logger
> 
> class FlagLogger:
>     """A wrapper to allow logger.info(flag, message) syntax."""
>     def __init__(self, logger):
>         self.logger = logger
> 
>     def info(self, flag, msg, *args, **kwargs):
>         self.logger.info(msg, *args, extra={"flag": flag}, **kwargs)
> 
>     def warning(self, flag, msg, *args, **kwargs):
>         self.logger.warning(msg, *args, extra={"flag": flag}, **kwargs)
> 
>     def error(self, flag, msg, *args, **kwargs):
>         self.logger.error(msg, *args, extra={"flag": flag}, **kwargs)
> 
>     def debug(self, flag, msg, *args, **kwargs):
>         self.logger.debug(msg, *args, extra={"flag": flag}, **kwargs)
> 
> ====================== INITIALIZE LOGGER ======================
> 
> def setup_logger(log_dir, log_name, logger_obj_name="logger_obj_name"):
>     """
>     Configure the logger system: output to both console and file simultaneously.
>     """
>     # 1. If the log directory doesn't exist, create it
>     if not os.path.exists(log_dir):
>         os.makedirs(log_dir)
>         print(f"Log directory created: {log_dir}")
> 
>     # 2. Generate a timestamped log filename, e.g., scraper_log_20231027_103000.log
>     log_filename = f"{log_name}-{datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
>     log_filepath = os.path.join(log_dir, log_filename)
> 
>     # 3. Create Logger object
>     logger = logging.getLogger(logger_obj_name)
>     logger.setLevel(logging.INFO) # Set the minimum logger level
>     logger.handlers = [] # Clear previous handlers to prevent duplicate printing
> 
>     # --- Define a unified format (time accurate to the second) ---
>     # %(asctime)s : Time
>     # %(levelname)s : Log level (INFO/ERROR)
>     # %(message)s : Your message content
>     file_formatter = logging.Formatter(
>         '[%(asctime)s][%(levelname)s][%(flag)s] %(message)s', 
>         datefmt="%y-%#m-%#d %H:%M:%S"
>     )
>     console_formatter = logging.Formatter(
>         '[%(asctime)s][%(levelname)s][%(flag)s] %(message)s', 
>         datefmt="%y-%#m-%#d %H:%M:%S"
>     )
> 
>     # --- Handler 1: File output (detailed, with timestamp) ---
>     file_handler = logging.FileHandler(log_filepath, encoding='utf-8')
>     file_handler.setFormatter(file_formatter)
>     logger.addHandler(file_handler)
> 
>     # --- Handler 2: Console output (concise, for human reading) ---
>     console_handler = logging.StreamHandler()
>     console_handler.setFormatter(console_formatter)
>     logger.addHandler(console_handler)
> 
>     # --- Return the Wrapped Logger ---
>     logger = FlagLogger(logger)
>     logger.info("SETUP LOG", f"✅ logger System Started")
>     logger.info("SETUP LOG", f"Log file path: {log_filepath}")
>     return logger
> ```

Because your `setup_logger` returns a custom wrapper class (`FlagLogger`) instead of a standard Python logger object, you have a unique constraint: **if Script B simply look up the logger via `logging.getLogger()`, it won't have your custom `.info(flag, msg)` syntax.** To get around this, you must ensure Script B uses a `FlagLogger` instance that points to Script A's underlying logging engine. Here are the two best ways to handle this with your exact setup.

---

## Method 1: The Named Singleton Trick (Recommended & Automatic)

Python's `logging.getLogger(name)` always returns the exact same underlying logger instance across your entire application. Script B can fetch that underlying logger by its string name and wrap it inside a fresh `FlagLogger`.

### Script B (`script_b.py`)

Use an `if __name__ == "__main__":` guard block. If imported, Script B hooks into Script A's named logger object and applies your `FlagLogger` wrapper to it.

```python
import logging
from utils.logger import setup_logger, FlagLogger

# Determine if we are running standalone or being imported
if __name__ == "__main__":
    # Run independently: create its own separate log file
    logger = setup_logger("logs/script_b", "script_b", logger_obj_name="standalone_b")
else:
    # Imported by Script A: Fetch A's underlying logger by name and wrap it
    # so the .info(flag, msg) syntax still works flawlessly!
    underlying_logger = logging.getLogger("shared_app_logger")
    logger = FlagLogger(underlying_logger)

def do_something():
    # This works perfectly whether running standalone or imported!
    logger.info("SCRIPT_B", "Executing tasks inside Script B...")

```

### Script A (`script_a.py`)

In your main script, pass a specific string to `logger_obj_name` so Script B knows what name to look for.

```python
from utils.logger import setup_logger
import script_b  # Safe to import anywhere

# Initialize the global logger with a specific object name
logger = setup_logger("logs/script_a", "main_app", logger_obj_name="shared_app_logger")

def main():
    logger.info("MAIN_A", "🚀 Script A has started.")
    
    # Call Script B (it will automatically write to Script A's log file)
    script_b.do_something()

if __name__ == "__main__":
    main()

```

> **Why this works seamlessly:** Even if Script B is imported *before* Script A runs `setup_logger`, Python creates a placeholder singleton for `"shared_app_logger"`. When Script A finally invokes `setup_logger`, it flushes and configures that exact same placeholder with your file/console handlers.

---

## Method 2: Direct Variable Injection (No Hardcoded Names)

If you don't want to rely on matching the string name (`"shared_app_logger"`) across multiple files, you can leave Script B's logger empty when imported, and let Script A explicitly pass its `FlagLogger` instance directly into Script B's namespace.

### Script B (`script_b.py`)

```python
from utils.logger import setup_logger

if __name__ == "__main__":
    # Only sets up a local file logger if run directly
    logger = setup_logger("logs/script_b", "script_b")
else:
    # Set as a placeholder when imported. Script A will fill this in.
    logger = None

def do_something():
    if logger:
        logger.info("SCRIPT_B", "Hello from Script B using injected logger!")
    else:
        print("Error: Logger was never shared by the parent script.")

```

### Script A (`script_a.py`)

```python
from utils.logger import setup_logger
import script_b

# 1. Initialize Script A's wrapper logger
logger = setup_logger("logs/script_a", "main_app")

# 2. Inject Script A's FlagLogger object straight into Script B
script_b.logger = logger

def main():
    logger.info("MAIN_A", "Script A running...")
    script_b.do_something()

if __name__ == "__main__":
    main()

```

> For Method 1 and for script b, if I use from script_b import fun, is it worked?

**Yes, it will work perfectly!** Whether you use `import script_b` or `from script_b import fun`, Python executes the exact same underlying process: it loads the entirety of `script_b.py` into memory from top to bottom. This means the global `else` block in Script B still runs, and its local `logger` variable is still created successfully.

Here is a step-by-step breakdown of how Python handles the timing behind the scenes to make sure it works:

### The Step-by-Step Execution Flow

1. **The Import:** In Script A, Python hits `from script_b import fun`. It pauses Script A and runs Script B's global space.
2. **The Wrapper Creation:** Script B sees it's being imported, hits the `else` block, and calls `logging.getLogger("shared_app_logger")`.
> *Note:* At this exact microsecond, `"shared_app_logger"` has no file or console handlers attached to it yet. Python just creates an empty "shell" logger object. Script B wraps this shell in `FlagLogger(underlying_logger)`.


3. **The Configuration:** Control returns to Script A. Script A now runs `logger = setup_logger(..., logger_obj_name="shared_app_logger")`. Inside your setup function, it grabs that exact same "shell" logger object from Python's internal registry and injects the file handlers and console handlers into it.
4. **The Execution:** When you finally call `fun()`, it uses the `FlagLogger` inside Script B. Because the underlying logger object is shared by reference, it now perfectly sees all the file and console handlers that Script A attached to it.

---

### ⚠️ One Important Rule to Keep in Mind

Because `from script_b import fun` usually happens at the very top of Script A, **Script B is imported before Script A actually configures the logger.**

Because of this, you must **only use the logger inside functions** in Script B.

#### 🛑 What will NOT work:

If you put a log message right in the open global space of Script B, it will try to log *before* Script A has attached the file handlers.

```python
# inside script_b.py
else:
    underlying_logger = logging.getLogger("shared_app_logger")
    logger = FlagLogger(underlying_logger)

logger.info("TEST", "This will NOT print to the file!") # ❌ Too early! Script A hasn't run setup_logger yet.

def fun():
    ...

```

#### What WILL work flawlessly:

```python
# inside script_b.py
def fun():
    #  Perfect! This only runs when called by Script A's main(), 
    # long after setup_logger has finished its job.
    logger.info("SCRIPT_B", "This prints to Script A's file perfectly!") 

```